# Feedlot V2 · Monitor Drive

Buka notebook ini di tab dan runtime terpisah dari produksi. Runtime CPU standar cukup.
Jalankan persiapan, hubungkan akun Drive yang dipakai saat START, lalu tempel ID folder.
Monitor hanya membaca Drive. Jalankan ulang sel terakhir untuk menyegarkan snapshot.

[Notebook produksi](https://colab.research.google.com/github/wijaya-rama/portfolio-feedlot-management/blob/stage7-colab-live-monitoring/simulator_v2/stage6/colab/Feedlot_V2_Production.ipynb) ·
[Panduan pemulihan](https://github.com/wijaya-rama/portfolio-feedlot-management/blob/stage7-colab-live-monitoring/simulator_v2/stage6/colab/MONITORING_PRODUKSI.md)

In [ ]:
# @title 1 · Persiapan Monitor Drive
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/wijaya-rama/portfolio-feedlot-management.git"
MONITOR_SHA = "22c6a2b47ca6433ddf13cb45a028d369142970a4"
ROOT = Path("/content/feedlot-v2-status")
SOURCE = ROOT / "source"
PYTHON = ROOT / "venv/bin/python"
SOURCE.mkdir(parents=True, exist_ok=True)

def git(*args):
    return subprocess.check_output(["git", "-C", str(SOURCE), *args], text=True).strip()

print("Menyiapkan Monitor Drive baca-saja.", flush=True)
if not (SOURCE / ".git").is_dir():
    git("init", "--quiet")
git("fetch", "--no-tags", "--depth=1", REPOSITORY, MONITOR_SHA)
assert git("rev-parse", "FETCH_HEAD") == MONITOR_SHA, "Commit hasil unduhan berbeda."
git("checkout", "--detach", "--quiet", MONITOR_SHA)
assert git("rev-parse", "HEAD") == MONITOR_SHA
assert not git("status", "--porcelain", "--untracked-files=no"), "Checkout sumber telah berubah."
sys.path.insert(0, str(SOURCE))
from simulator_v2.stage6.colab.notebook_stream import stream_command

for command in (
    [sys.executable, "-m", "venv", str(ROOT / "venv")],
    [str(PYTHON), "-m", "pip", "install", "--quiet", "google-api-python-client==2.202.0"],
):
    if stream_command(command) != 0:
        raise RuntimeError("Persiapan Monitor Drive gagal.")
print("Monitor siap. Hubungkan Drive.", flush=True)

In [ ]:
# @title 2 · Hubungkan Drive
from google.colab import auth

auth.authenticate_user()
print("Otorisasi selesai. Masukkan ID folder di sel berikut.", flush=True)

In [ ]:
# @title 3 · Baca status dan verifikasi metadata
ID_folder_Drive = ""  # @param {type:"string"}

if ID_folder_Drive.strip():
    print("Memeriksa status, klaim dan checksum server Drive.", flush=True)
    code = stream_command([str(PYTHON), "-u", "-m", "simulator_v2.stage6.colab.drive_status",
                           ID_folder_Drive.strip()], cwd=SOURCE)
    if code:
        raise RuntimeError("Diagnostik Drive gagal. Periksa output di atas dan akun yang digunakan.")
else:
    print("Masukkan ID dari tautan folder percobaan Drive.")

`last_heartbeat` memuat waktu UTC, fase dan waktu yang sudah berjalan jika terekam.
`claim_lock.state=unreleased` bisa berarti worker masih aktif atau klaim tertinggal.
`last_verified_checkpoint` hanya mencakup arsip yang sudah terbit dan lolos pemeriksaan metadata Drive.
Bila `issues` terisi, simpan hasil diagnostik untuk pemeriksaan. Monitor tidak mengambil alih klaim.